In [1]:
from pyspark.sql import SparkSession

# Build the SparkSession with the MongoDB Connector
spark = SparkSession.builder \
    .appName("MongoDB_PySpark_Integration") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Spark Session with MongoDB Connector created successfully!")

Spark Version: 3.3.0
Spark Session with MongoDB Connector created successfully!


In [7]:
# Read the HDFS file
movies_df = spark.read \
    .option("sep", "::") \
    .option("inferSchema", "true") \
    .csv("hdfs://namenode:9000/user/root/movielens/movies.dat") \
    .toDF("MovieID", "Title", "Genres")

print("Data loaded from HDFS. Total records:", movies_df.count())

Data loaded from HDFS. Total records: 3883


In [5]:
# Write to MongoDB
movies_df.write \
    .format("mongodb") \
    .mode("overwrite") \
    .option("spark.mongodb.connection.uri", "mongodb://mongodb:27017/") \
    .option("spark.mongodb.database", "movielens_db") \
    .option("spark.mongodb.collection", "movies") \
    .save()

print("Data successfully written to MongoDB!")

Data successfully written to MongoDB!


In [6]:
# Read back from MongoDB
mongo_movies_df = spark.read \
    .format("mongodb") \
    .option("spark.mongodb.connection.uri", "mongodb://mongodb:27017/") \
    .option("spark.mongodb.database", "movielens_db") \
    .option("spark.mongodb.collection", "movies") \
    .load()

# Show the schema and data
mongo_movies_df.printSchema()
mongo_movies_df.show(5, truncate=False)

root
 |-- Genres: string (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- _id: string (nullable = true)

+----------------------------+-------+----------------------------------+------------------------+
|Genres                      |MovieID|Title                             |_id                     |
+----------------------------+-------+----------------------------------+------------------------+
|Animation|Children's|Comedy |1      |Toy Story (1995)                  |69e1e72831778752c7ad4098|
|Adventure|Children's|Fantasy|2      |Jumanji (1995)                    |69e1e72831778752c7ad4099|
|Comedy|Romance              |3      |Grumpier Old Men (1995)           |69e1e72831778752c7ad409a|
|Comedy|Drama                |4      |Waiting to Exhale (1995)          |69e1e72831778752c7ad409b|
|Comedy                      |5      |Father of the Bride Part II (1995)|69e1e72831778752c7ad409c|
+----------------------------+-------+--------------

In [8]:
!pip install pymongo


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.8 MB/s eta 0:00:00a 0:00:01


In [9]:
from pymongo import MongoClient

# 1. Connect directly to the MongoDB container
client = MongoClient("mongodb://mongodb:27017/")

# 2. Select the database and collection
db = client["movielens_db"]
collection = db["movies"]

# 3. Create a NoSQL JSON query (Find all movies containing 'Comedy' in the Genres)
query = {"Genres": {"$regex": "Comedy"}}

# 4. Execute the query and limit to 3 results
results = collection.find(query).limit(3)

print("--- Native MongoDB JSON Documents ---")
for doc in results:
    print(doc)

--- Native MongoDB JSON Documents ---
{'_id': ObjectId('69e1e72831778752c7ad4098'), 'MovieID': 1, 'Title': 'Toy Story (1995)', 'Genres': "Animation|Children's|Comedy"}
{'_id': ObjectId('69e1e72831778752c7ad409a'), 'MovieID': 3, 'Title': 'Grumpier Old Men (1995)', 'Genres': 'Comedy|Romance'}
{'_id': ObjectId('69e1e72831778752c7ad409b'), 'MovieID': 4, 'Title': 'Waiting to Exhale (1995)', 'Genres': 'Comedy|Drama'}


In [10]:
# Stop the Spark session
spark.stop()
print("Spark Session Stopped.")

Spark Session Stopped.
